<a href="https://colab.research.google.com/github/Bunkhuoch-Ann/Side_Quests_simple/blob/main/Dark_matter_and_RC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initialization

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# IMPORTS & GLOBAL CONSTANTS
# ──────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless rendering (safe for saving MP4s in a notebook)
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from tqdm.notebook import tqdm

G = 1.0             # gravitational constant, natural units (like the reference sim)
SOFTENING = 0.02     # softens the 1/r^2 singularity right at the center

# A "solar system" palette — one color per orbiting body
COLORS = ["#f59e0b", "#ec4899", "#3b82f6", "#10b981", "#a855f7", "#f43f5e", "#22d3ee"]


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# PHYSICS PART 1: central point-mass (Keplerian) gravity
# ──────────────────────────────────────────────────────────────────────────
def a_central(pos, M_central, eps=SOFTENING):
    '''pos: (...,3) array of positions. Returns acceleration toward the origin.'''
    r = np.linalg.norm(pos, axis=-1, keepdims=True)
    return -G * M_central * pos / (r ** 2 + eps ** 2) ** 1.5


def phi_central(r, M_central, eps=SOFTENING):
    '''Potential of the central point mass (softened).'''
    return -G * M_central / np.sqrt(r ** 2 + eps ** 2)


def v_keplerian(r, M_central):
    '''Circular-orbit speed under central gravity alone.'''
    return np.sqrt(G * M_central / np.maximum(r, 1e-6))


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# PHYSICS PART 2: dark matter halo (pseudo-isothermal sphere -> flat curve)
# ──────────────────────────────────────────────────────────────────────────
def a_dm(pos, v_halo, r_c):
    '''Acceleration from the dark-matter halo (pulls toward the origin).'''
    r2 = np.sum(pos ** 2, axis=-1, keepdims=True)
    return -v_halo ** 2 * pos / (r2 + r_c ** 2)


def phi_dm(r, v_halo, r_c):
    '''Potential of the dark-matter halo.'''
    return 0.5 * v_halo ** 2 * np.log(r ** 2 + r_c ** 2)


def rho_dm(r, v_halo, r_c):
    '''Dark-matter mass density (for the heat map), from Poisson's equation.'''
    return (v_halo ** 2 / (4 * np.pi * G)) * (r ** 2 + 3 * r_c ** 2) / (r ** 2 + r_c ** 2) ** 2


def v_flat_curve(r, M_central, v_halo, r_c):
    '''Circular-orbit speed under central + DM gravity together.
    -> declining near the center (central mass dominates), flat at large r (halo dominates).'''
    a_mag = G * M_central / np.maximum(r, 1e-6) ** 2 + v_halo ** 2 * r / (r ** 2 + r_c ** 2)
    return np.sqrt(r * a_mag)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# INITIAL CONDITIONS
# ──────────────────────────────────────────────────────────────────────────
def build_initial_conditions(radii, M_central, v_halo, r_c, mode="keplerian",
                              inclination_deg=4.0, ellipticity_factor=1.0, seed=42):
    '''Place N test-particle 'planets' on near-circular orbits at the given radii.

    mode: 'keplerian'          -> Scenario A speeds
          'dm_flat'             -> Scenario B speeds (matches the real potential)
          'forced_flat_no_dm'   -> Scenario C speeds (mismatched on purpose)
    ellipticity_factor: A factor (0 to 1) to scale the tangential velocity. 1.0 for circular,
                        <1.0 for elliptical orbits (initial position is apocenter).
    '''
    rng = np.random.default_rng(seed)
    n = len(radii)
    radii = np.asarray(radii, dtype=float)

    if mode == "keplerian":
        speed = v_keplerian(radii, M_central)
    elif mode in ("dm_flat", "forced_flat_no_dm"):
        speed = v_flat_curve(radii, M_central, v_halo, r_c)
    else:
        raise ValueError(f"unknown mode: {mode}")

    pos = np.zeros((n, 3))
    vel = np.zeros((n, 3))
    for i, r in enumerate(radii):
        theta = rng.uniform(0, 2 * np.pi)               # random starting angle
        incl = np.deg2rad(inclination_deg) * rng.uniform(-1, 1)  # small tilt

        # position in the tilted orbital plane
        x, y = r * np.cos(theta), r * np.sin(theta)
        z = 0.0
        pos[i] = [x, y, z]

        # circular velocity, perpendicular to the radius vector, then tilt by `incl`
        # Apply ellipticity_factor here
        vx, vy = -speed[i] * ellipticity_factor * np.sin(theta), speed[i] * ellipticity_factor * np.cos(theta)
        vz = speed[i] * ellipticity_factor * np.sin(incl) # vz also scaled by ellipticity factor
        vy *= np.cos(incl)
        vel[i] = [vx, vy, vz]

    masses = np.full(n, 1e-4)  # test-particle masses (only used for display/energy bookkeeping)
    return pos, vel, masses

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# PHYSICS: RK4 integrator
# ──────────────────────────────────────────────────────────────────────────
def accelerations(pos, M_central, v_halo, r_c, dm_enabled):
    '''pos: (n,3) positions of all planets. Returns (n,3) accelerations.
    Planets are test particles -> no planet-planet gravity, only the external
    potential (central mass, plus the halo if dm_enabled).'''
    acc = a_central(pos, M_central)
    if dm_enabled:
        acc = acc + a_dm(pos, v_halo, r_c)
    return acc


def rk4_step(pos, vel, M_central, v_halo, r_c, dm_enabled, dt):
    def deriv(p, v):
        return v, accelerations(p, M_central, v_halo, r_c, dm_enabled)

    k1p, k1v = deriv(pos, vel)
    k2p, k2v = deriv(pos + 0.5 * dt * k1p, vel + 0.5 * dt * k1v)
    k3p, k3v = deriv(pos + 0.5 * dt * k2p, vel + 0.5 * dt * k2v)
    k4p, k4v = deriv(pos + dt * k3p, vel + dt * k3v)

    new_pos = pos + (dt / 6.0) * (k1p + 2 * k2p + 2 * k3p + k4p)
    new_vel = vel + (dt / 6.0) * (k1v + 2 * k2v + 2 * k3v + k4v)
    return new_pos, new_vel


def total_energy(pos, vel, masses, M_central, v_halo, r_c, dm_enabled):
    '''Per-particle KE + PE in the external potential, summed. A conserved quantity
    when dm_enabled matches what was actually used to integrate -> our sanity check.'''
    r = np.linalg.norm(pos, axis=-1)
    ke = 0.5 * masses * np.sum(vel ** 2, axis=-1)
    pe = masses * phi_central(r, M_central)
    if dm_enabled:
        pe = pe + masses * phi_dm(r, v_halo, r_c)
    return np.sum(ke + pe)


In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# RUN SIMULATION
# ──────────────────────────────────────────────────────────────────────────
def run_simulation(pos0, vel0, masses, M_central, v_halo, r_c, dm_enabled,
                    dt=0.01, n_steps=4000, record_every=2):
    pos = np.copy(pos0)
    vel = np.copy(vel0)

    traj_list = [np.copy(pos)]
    vel_list = [np.copy(vel)]  # New: Store velocities
    energy_list = [total_energy(pos, vel, masses, M_central, v_halo, r_c, dm_enabled)]
    times_list = [0.0]

    t = 0.0
    for step in tqdm(range(n_steps), desc="RK4 integration"):
        pos, vel = rk4_step(pos, vel, M_central, v_halo, r_c, dm_enabled, dt)
        t += dt
        if (step + 1) % record_every == 0:
            traj_list.append(np.copy(pos))
            vel_list.append(np.copy(vel)) # New: Store velocities
            energy_list.append(total_energy(pos, vel, masses, M_central, v_halo, r_c, dm_enabled))
            times_list.append(t)

    return np.array(traj_list), np.array(energy_list), np.array(times_list), np.array(vel_list), pos, vel


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless rendering (safe for saving MP4s in a notebook)
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from tqdm.notebook import tqdm

G = 1.0             # gravitational constant, natural units (like the reference sim)
SOFTENING = 0.02     # softens the 1/r^2 singularity right at the center

# A "solar system" palette — one color per orbiting body
COLORS = ["#f59e0b", "#ec4899", "#3b82f6", "#10b981", "#a855f7", "#f43f5e", "#22d3ee"]

def render_video(traj, energy, times, masses, M_central, v_halo, r_c, dm_enabled,
                  output_path, title, radii_for_scale, trail_len=400, fps=30,
                  n_frames_out=400, unbound_flag=False, show_rotation_curve=False,
                  R_max_override=None, vel_s=None, dm_enabled_per_frame=None,
                  plot_main_only=False, view_inclination_deg=0.0): # New parameters

    def col(i):
        return COLORS[i % len(COLORS)]

    n_total = traj.shape[0]
    idx = np.linspace(0, n_total - 1, min(n_frames_out, n_total)).astype(int)

    # Sampled physical (unrotated) data
    traj_physical_sampled = traj[idx]
    energy_s = energy[idx]
    times_s = times[idx]
    vel_physical_sampled = vel_s[idx] if vel_s is not None else None
    n_bodies = traj.shape[1]

    # `traj_view` will hold the positions after applying camera rotation for display
    traj_view = np.copy(traj_physical_sampled)

    # --- Apply camera rotation to positions if specified ---
    if view_inclination_deg != 0.0:
        view_inclination_rad = np.deg2rad(view_inclination_deg)
        # Rotation matrix around X-axis for viewing transformation
        cos_alpha = np.cos(view_inclination_rad)
        sin_alpha = np.sin(view_inclination_rad)
        rotation_matrix_view = np.array([
            [1, 0, 0],
            [0, cos_alpha, sin_alpha],
            [0, -sin_alpha, cos_alpha]
        ])
        traj_view = np.einsum('ijk,kl->ijl', traj_view, rotation_matrix_view)

    # No need for v_phi_data storing v_phi vs time anymore

    # --- Figure and subplot setup ---
    if plot_main_only:
        fig = plt.figure(figsize=(7, 7), facecolor="#050a12") # Adjust size for single panel
        gs = None # No grid spec needed for a single plot
        ax_main = fig.add_subplot(1, 1, 1, facecolor="#050a12")
        ax_side = None # No side panel
        ax_energy = None # No energy panel
        # Override general figure title with specific one for main panel if only main is plotted
        ax_main.set_title(title, color="#e2e8f0", fontsize=14, loc="left", fontweight="bold")
    else:
        fig = plt.figure(figsize=(13, 7), facecolor="#050a12") # Original size
        gs = fig.add_gridspec(2, 3, width_ratios=[2.2, 1, 1], height_ratios=[1, 1],
                               hspace=0.35, wspace=0.3)
        ax_main = fig.add_subplot(gs[:, 0], facecolor="#050a12")
        ax_side = fig.add_subplot(gs[0, 1:], facecolor="#0a111c")
        ax_energy = fig.add_subplot(gs[1, 1:], facecolor="#0a111c")
        ax_main.set_title(title, color="#e2e8f0", fontsize=11, loc="left", fontweight="bold")

    # ---- plot extent, generous enough to hold the outermost orbit or the unbound escape
    R_max = R_max_override if R_max_override is not None else (max(radii_for_scale) * (2.2 if unbound_flag else 1.4))

    # ---- background heat map: dark-matter density
    im = None # Initialize im to None
    im_should_be_created = ax_main is not None and (dm_enabled or (dm_enabled_per_frame is not None and np.any(dm_enabled_per_frame)))

    if im_should_be_created:
        grid_n = 220
        gx = np.linspace(-R_max, R_max, grid_n)
        GX, GY = np.meshgrid(gx, gx)
        GR = np.sqrt(GX ** 2 + GY ** 2)
        field = rho_dm(GR, v_halo, r_c)
        cmap = LinearSegmentedColormap.from_list("dm", ["#050a12", "#1e1b4b", "#7c3aed", "#f0abfc"])
        field_disp = np.log10(field + field.max() * 1e-6)
        im = ax_main.imshow(field_disp, extent=[-R_max, R_max, -R_max, R_max], origin="lower",
                             cmap=cmap, alpha=0.85, aspect="equal")
        # No colorbar for DM field as per user's request.

    if ax_main is not None:
        ax_main.scatter([0], [0], marker="*", s=100, color="#fde68a", edgecolor="white",
                         linewidth=0.5, zorder=5)  # the central mass
        ax_main.set_xlim(-R_max, R_max); ax_main.set_ylim(-R_max, R_max)
        ax_main.set_xlabel("x", color="#64748b"); ax_main.set_ylabel("y", color="#64748b")
        ax_main.tick_params(colors="#475569", labelsize=7)
        ax_main.set_aspect("equal", adjustable="box") # Ensure main panel is square
        for s in ax_main.spines.values(): s.set_color("#1e293b")

        trails_main = [ax_main.plot([], [], color=col(i), lw=1.0, alpha=0.85)[0] for i in range(n_bodies)]
        points_main = [ax_main.plot([], [], "o", color=col(i), markersize=6,
                                     markeredgecolor="white", markeredgewidth=0.4)[0] for i in range(n_bodies)]
    else:
        trails_main = []
        points_main = []

    # ---- edge-on (x-z) panel
    if ax_side is not None:
        ax_side.set_title("Edge-on view (X-Z)", color="#94a3b8", fontsize=9, loc="left")
        ax_side.set_xlim(-R_max, R_max); ax_side.set_ylim(-R_max * 0.35, R_max * 0.35)
        ax_side.tick_params(colors="#475569", labelsize=7)
        for s in ax_side.spines.values(): s.set_color("#1e293b")
        ax_side.set_aspect("equal")
        ax_side.axhline(0, color="#1e293b", lw=0.8)

        trails_side = [ax_side.plot([], [], color=col(i), lw=0.9, alpha=0.85)[0] for i in range(n_bodies)]
        points_side = [ax_side.plot([], [], "o", color=col(i), markersize=5)[0] for i in range(n_bodies)]
    else:
        trails_side = []
        points_side = []

    # ---- energy / azimuthal velocity panel
    info_panel_lines = [] # Initialize here to collect all lines/scatters

    if ax_energy is not None:
        if show_rotation_curve:
            if vel_physical_sampled is None:
                print("Warning: 'vel_s' (sampled velocities) not provided. Rotation curve plot cannot be generated.")
                show_rotation_curve = False # Fallback to energy plot if data is missing
            else:
                ax_energy.set_title("Azimuthal Velocity Profile v_φ(r)", color="#94a3b8", fontsize=9, loc="left")
                ax_energy.set_xlabel("Orbital Radius r", color="#64748b", fontsize=8)
                ax_energy.set_ylabel("Azimuthal Velocity v_φ", color="#64748b", fontsize=8)

                # Plot theoretical curves as background for the rotation curve panel
                r_profile_plot = np.linspace(0.5, R_max, 300)
                v_kepler_profile = v_keplerian(r_profile_plot, M_central)
                v_flat_dm_profile = v_flat_curve(r_profile_plot, M_central, v_halo, r_c)
                ax_energy.plot(r_profile_plot, v_kepler_profile, color="#f59e0b", lw=1.0, linestyle='--', alpha=0.7, label="Keplerian (theory)")
                ax_energy.plot(r_profile_plot, v_flat_dm_profile, color="#7c3aed", lw=1.0, linestyle='--', alpha=0.7, label="DM Flat (theory)")
                ax_energy.legend(loc='upper right', facecolor="#0a111c", edgecolor="#1e293b", labelcolor="#e2e8f0", fontsize=7)

                # Initialize scatter plot for current particle velocities
                scatters = []
                for i in range(n_bodies):
                    scatter, = ax_energy.plot([], [], 'o', color=col(i), markersize=5, zorder=5)
                    scatters.append(scatter)
                info_panel_lines.extend(scatters) # Add these scatter plot artists to info_panel_lines

                ax_energy.set_xlim(0, R_max) # Consistent x-axis for radius

        if not show_rotation_curve: # This block runs if show_rotation_curve was False or set to False above
            ax_energy.set_title("Total energy (conservation check)", color="#94a3b8", fontsize=9, loc="left")
            ax_energy.set_ylabel("Total Energy", color="#64748b", fontsize=8)
            e_pad = 0.05 * (np.ptp(energy_s) + 1e-9)
            ax_energy.set_ylim(energy_s.min() - e_pad, energy_s.max() + e_pad)
            ax_energy.set_xlim(times_s[0], times_s[-1])

            energy_line, = ax_energy.plot([], [], color="#60a5fa", lw=1.2)
            info_panel_lines.append(energy_line)

        ax_energy.tick_params(colors="#475569", labelsize=7)
        for s in ax_energy.spines.values(): s.set_color("#1e293b")
        if not show_rotation_curve: # Only set x-label to 't' if showing energy
            ax_energy.set_xlabel("t", color="#64748b", fontsize=8)

    time_text = fig.text(0.02, 0.95, "", color="#e2e8f0", fontsize=12, fontweight="bold", family="monospace")
    status_text = fig.text(0.02, 0.91, "", color="#f43f5e", fontsize=9, family="monospace")

    def init():
        artists = []
        if ax_main is not None: artists.extend(trails_main + points_main)
        if ax_side is not None: artists.extend(trails_side + points_side)
        if ax_energy is not None: artists.extend(info_panel_lines)
        artists.extend([time_text, status_text])
        if im is not None:
            artists.append(im)
        return artists

    def update(frame_i):
        lo = max(0, frame_i - trail_len)

        if ax_main is not None:
            for i in range(n_bodies):
                seg = traj_view[lo:frame_i + 1, i, :]
                trails_main[i].set_data(seg[:, 0], seg[:, 1])
                points_main[i].set_data([traj_view[frame_i, i, 0]], [traj_view[frame_i, i, 1]])

            # Handle dynamic DM field visibility
            if im is not None:
                original_frame_index = idx[frame_i]
                if dm_enabled_per_frame is not None and not dm_enabled_per_frame[original_frame_index]:
                    im.set_alpha(0) # Hide DM field
                else: # If dm_enabled_per_frame is None, or it's True for this frame
                    im.set_alpha(0.85) # Show DM field

        if ax_side is not None:
            for i in range(n_bodies):
                seg = traj_view[lo:frame_i + 1, i, :]
                trails_side[i].set_data(seg[:, 0], seg[:, 2])
                points_side[i].set_data([traj_view[frame_i, i, 0]], [traj_view[frame_i, i, 2]])

        if ax_energy is not None and show_rotation_curve:
            # Calculate current r and v_phi for all particles using 3D vectors from PHYSICAL data
            pos_current_3d = traj_physical_sampled[frame_i, :, :]
            vel_current_3d = vel_physical_sampled[frame_i, :, :]

            r_current_3d = np.linalg.norm(pos_current_3d, axis=-1)
            # Use np.where to handle division by zero for particles exactly at the origin
            r_current_3d_safe = np.where(r_current_3d == 0, 1e-6, r_current_3d)

            v_current_3d_mag_sq = np.sum(vel_current_3d**2, axis=-1)
            v_radial_current_sq = (np.sum(pos_current_3d * vel_current_3d, axis=-1) / r_current_3d_safe)**2

            # Ensure v_phi_sq is not negative due to floating point inaccuracies
            v_phi_current = np.sqrt(np.maximum(0, v_current_3d_mag_sq - v_radial_current_sq))

            # Update scatter plot data
            for i in range(n_bodies):
                scatters[i].set_data([r_current_3d[i]], [v_phi_current[i]])

            # Dynamically adjust y-limits, considering theoretical curves
            vphi_min_current, vphi_max_current = np.min(v_phi_current), np.max(v_phi_current)
            vphi_min_theory = min(v_kepler_profile.min(), v_flat_dm_profile.min()) # Already defined in init
            vphi_max_theory = max(v_kepler_profile.max(), v_flat_dm_profile.max()) # Already defined in init

            overall_min_vphi = min(vphi_min_current, vphi_min_theory)
            overall_max_vphi = max(vphi_max_current, vphi_max_theory)

            vphi_range = overall_max_vphi - overall_min_vphi if overall_max_vphi != overall_min_vphi else 1.0
            vphi_pad = 0.1 * vphi_range # More padding for rotation curve
            ax_energy.set_ylim(overall_min_vphi - vphi_pad, overall_max_vphi + vphi_pad)

        elif ax_energy is not None: # Plot energy
            info_panel_lines[0].set_data(times_s[:frame_i + 1], energy_s[:frame_i + 1])

        time_text.set_text(f"t = {times_s[frame_i]:.2f}")

        if unbound_flag:
            max_r = np.linalg.norm(traj_physical_sampled[frame_i], axis=-1).max() # Use physical traj for unbound check
            if max_r > max(radii_for_scale) * 1.5:
                status_text.set_text("⚠ system flying apart — unbound orbits")
            else:
                status_text.set_text("")
        else:
            status_text.set_text("")
        artists = []
        if ax_main is not None: artists.extend(trails_main + points_main)
        if ax_side is not None: artists.extend(trails_side + points_side)
        if ax_energy is not None: artists.extend(info_panel_lines)
        artists.extend([time_text, status_text])
        if im is not None:
            artists.append(im)
        return artists

    anim = animation.FuncAnimation(fig, update, frames=tqdm(range(len(traj_physical_sampled)), desc="Rendering video"), init_func=init,
                                blit=False, interval=1000 / fps)
    writer = animation.FFMpegWriter(fps=fps, bitrate=4000)
    anim.save(output_path, writer=writer, dpi=130)
    plt.close(fig)
    print(f"Saved -> {output_path}")

## Normal Solar system

In [ ]:
# Shared "solar system" layout: 6 planets at increasing radii
RADII = np.array([2.0, 3.5, 5.5, 8.0, 11.5, 15.5])
M_CENTRAL = 40.0     # mass of the central "sun"
V_HALO = 3.2         # dark-matter halo's asymptotic flat speed
R_CORE = 4.0         # dark-matter halo's core radius

pos0_A, vel0_A, masses_A = build_initial_conditions(
    RADII, M_CENTRAL, V_HALO, R_CORE, mode="keplerian", inclination_deg=4, ellipticity_factor=0.975)

traj_A, energy_A, times_A, vel_A, pos_A_final, vel_A_final = run_simulation(
    pos0_A, vel0_A, masses_A, M_CENTRAL, V_HALO, R_CORE, dm_enabled=False,
    dt=0.001, n_steps=40000, record_every=2)

render_video(traj_A, energy_A, times_A, masses_A, M_CENTRAL, V_HALO, R_CORE,
             dm_enabled=False, output_path="scenario_A_keplerian.mp4",
             title="A. Normal Solar System — Keplerian decline",
             radii_for_scale=RADII, unbound_flag=False,
             R_max_override=RADII[-1]*1.4, vel_s=vel_A,
             dm_enabled_per_frame=np.full(len(traj_A), False),
             show_rotation_curve=True, plot_main_only=True, view_inclination_deg=0)

RK4 integration:   0%|          | 0/40000 [00:00<?, ?it/s]

Rendering video:   0%|          | 0/400 [00:00<?, ?it/s]

Saved -> scenario_A_keplerian.mp4


## Dark matter halo and flat RC

In [ ]:
pos0_B, vel0_B, masses_B = build_initial_conditions(
    RADII, M_CENTRAL, V_HALO, R_CORE, mode="dm_flat", inclination_deg=4.0, ellipticity_factor=0.975)

traj_B, energy_B, times_B, vel_B, pos_B_final, vel_B_final = run_simulation(
    pos0_B, vel0_B, masses_B, M_CENTRAL, V_HALO, R_CORE, dm_enabled=True,
    dt=0.001, n_steps=40000, record_every=2)

render_video(traj_B, energy_B, times_B, masses_B, M_CENTRAL, V_HALO, R_CORE,
             dm_enabled=True, output_path="scenario_B_dark_matter_flat.mp4",
             title="B. With dark matter halo — flat rotation curve",
             radii_for_scale=RADII, unbound_flag=False,
             R_max_override=RADII[-1]*1.4, vel_s=vel_B,
             dm_enabled_per_frame=np.full(len(traj_B), True),
             show_rotation_curve=True, plot_main_only=True, view_inclination_deg=0.0) # Keep energy plot

RK4 integration:   0%|          | 0/40000 [00:00<?, ?it/s]

Rendering video:   0%|          | 0/400 [00:00<?, ?it/s]

Saved -> scenario_B_dark_matter_flat.mp4


## No DM but force flat RC

In [ ]:
pos0_C, vel0_C, masses_C = build_initial_conditions(
    RADII, M_CENTRAL, V_HALO, R_CORE, mode="forced_flat_no_dm", inclination_deg=4.0, ellipticity_factor=0.975)

traj_C, energy_C, times_C, vel_C, pos_C_final, vel_C_final = run_simulation(
    pos0_C, vel0_C, masses_C, M_CENTRAL, V_HALO, R_CORE, dm_enabled=False,
    dt=0.001, n_steps=40000, record_every=2)

render_video(traj_C, energy_C, times_C, masses_C, M_CENTRAL, V_HALO, R_CORE,
             dm_enabled=False, output_path="scenario_C_flat_no_dm_flies_apart.mp4",
             title="C. Flat speeds, NO dark matter — flies apart!",
             radii_for_scale=RADII, unbound_flag=True,
             R_max_override=RADII[-1]*2.2, vel_s=vel_C,
             dm_enabled_per_frame=np.full(len(traj_C), False),
             show_rotation_curve=True, plot_main_only=True, view_inclination_deg=0.0) # Keep energy plot

RK4 integration:   0%|          | 0/40000 [00:00<?, ?it/s]

Rendering video:   0%|          | 0/400 [00:00<?, ?it/s]

Saved -> scenario_C_flat_no_dm_flies_apart.mp4


### Scenario D: Continue from B, but turn off Dark Matter

This scenario demonstrates what happens if a system, initially in equilibrium with a dark matter halo (like Scenario B), suddenly has its dark matter removed. The planets, which were on stable flat rotation curves, are now moving too fast for a purely Keplerian potential and will tend to fly apart. We'll use the final state of Scenario B as the initial state for this new phase.

In [ ]:
# D. Continue from Scenario B (dark matter flat rotation curve) but turn off DM.
# This simulates what happens if the dark matter 'disappears'.

dm_enabled_D = False # Explicitly turn off dark matter
n_steps_D = 40000     # Run for the same number of steps as previous scenarios
dt_D = 0.001          # Same time step
record_every_D = 2   # Same recording frequency

# Run the simulation for Scenario D, using final conditions of B as initial conditions
traj_D, energy_D, times_D_raw, vel_D, _, _ = run_simulation(
    pos_B_final, vel_B_final, masses_B, M_CENTRAL, V_HALO, R_CORE, dm_enabled=dm_enabled_D,
    dt=dt_D, n_steps=n_steps_D, record_every=record_every_D)

# Adjust times for Scenario D to follow Scenario B continuously
times_D_offset = times_B[-1] + times_D_raw # Add the final time of B to all times in D

# Concatenate results of Scenario B and Scenario D to create a single continuous simulation
# Skip the first element of traj_D, energy_D, and vel_D to avoid duplicating the final state of B
traj_BD = np.concatenate((traj_B, traj_D[1:]), axis=0)
energy_BD = np.concatenate((energy_B, energy_D[1:]), axis=0)
times_BD = np.concatenate((times_B, times_D_offset[1:]), axis=0) # Skip first time point of D to avoid duplicate
vel_BD = np.concatenate((vel_B, vel_D[1:]), axis=0)

# Create the dm_enabled_per_frame array for Scenario D
dm_flags_B = np.full(len(traj_B), True, dtype=bool)
dm_flags_D_no_dm = np.full(len(traj_D[1:]), False, dtype=bool) # Adjusted length
dm_enabled_for_frames_BD = np.concatenate((dm_flags_B, dm_flags_D_no_dm))

# Render the combined video, demonstrating azimuthal velocity profile and consistent scaling
render_video(traj_BD, energy_BD, times_BD, masses_B, M_CENTRAL, V_HALO, R_CORE,
             dm_enabled=True, # Set to True to ensure 'im' is created
             output_path="scenario_D_B_then_no_dm.mp4",
             title="D. B then turn-off Dark Matter",
             radii_for_scale=RADII,
             unbound_flag=True, # The system is expected to become unbound
             show_rotation_curve=True, # Show azimuthal velocity PROFILE instead of energy
             R_max_override=RADII[-1]*2.2, # Consistent scaling with unbound C case
             vel_s=vel_BD,
             dm_enabled_per_frame=dm_enabled_for_frames_BD, plot_main_only=True, view_inclination_deg=0.0) # Pass the new flag array

RK4 integration:   0%|          | 0/40000 [00:00<?, ?it/s]

Rendering video:   0%|          | 0/400 [00:00<?, ?it/s]

Saved -> scenario_D_B_then_no_dm.mp4


## Bonus plot

In [ ]:
r_plot = np.linspace(0.5, max(RADII) * 1.3, 300)
v_kepler_curve = v_keplerian(r_plot, M_CENTRAL)
v_flat_curve_vals = v_flat_curve(r_plot, M_CENTRAL, V_HALO, R_CORE)

fig, ax = plt.subplots(figsize=(7, 4.5), facecolor="#050a12")
ax.set_facecolor("#0a111c")
ax.plot(r_plot, v_kepler_curve, color="#f59e0b", lw=2, label="A: Keplerian (no dark matter)")
ax.plot(r_plot, v_flat_curve_vals, color="#7c3aed", lw=2, label="B: with dark matter (flat)")
ax.scatter(RADII, v_flat_curve(RADII, M_CENTRAL, V_HALO, R_CORE), color="#f43f5e",
           zorder=5, label="C: planets' actual speed\n(but only A's gravity applies)")
ax.set_xlabel("orbital radius r", color="#94a3b8")
ax.set_ylabel("circular speed v(r)", color="#94a3b8")
ax.set_title("Rotation curves: Keplerian decline vs. flat (dark matter)", color="#e2e8f0")
ax.tick_params(colors="#475569")
for s in ax.spines.values(): s.set_color("#1e293b")
leg = ax.legend(facecolor="#0a111c", edgecolor="#1e293b", labelcolor="#e2e8f0", fontsize=8)
plt.tight_layout()
plt.savefig("rotation_curves_comparison.png", dpi=150, facecolor=fig.get_facecolor())
plt.show()
